# Phase 3 Setup — AWS Resources

Creates all five AWS resources needed for the automated Phase 1 → Phase 2 chain:

```
S3 ObjectCreated → Lambda (batch-manifest-writer)
  → DynamoDB posts_batch_manifest (status="uploaded")
  → SQS posts-batch-queue
  → Phase 2 spark-submit --mode sqs
  → DynamoDB (status="done")
  → SNS email
```

**Run from inside `scripts/`** (so Cell 4 can read `lambda_function.py`).

Cells are idempotent: re-running after a partial failure is safe.

In [1]:
import boto3
import io
import json
import os
import zipfile

REGION                 = "us-east-1"
BUCKET                 = "se-data-dump-2025-3"
DYNAMODB_TABLE         = "posts_batch_manifest"
SQS_QUEUE_NAME         = "posts-batch-queue"
LAMBDA_NAME            = "batch-manifest-writer"
SNS_TOPIC_NAME         = "batch-etl-complete"
NOTIFY_EMAIL           = "graceeehgq@gmail.com"
SQS_VISIBILITY_TIMEOUT = 900   # seconds — must be > one batch Spark processing time
LAMBDA_TIMEOUT         = 30    # seconds

session    = boto3.Session(region_name=REGION)
sts        = session.client("sts")
dynamodb_c = session.client("dynamodb")
sqs        = session.client("sqs")
lam        = session.client("lambda")
s3         = session.client("s3")
sns        = session.client("sns")

ACCOUNT_ID   = sts.get_caller_identity()["Account"]
LAB_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/LabRole"
LAMBDA_ARN   = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"

print(f"Account ID : {ACCOUNT_ID}")
print(f"S3 bucket  : {BUCKET}")
print(f"IAM role   : {LAB_ROLE_ARN}")

Account ID : 650720751298
S3 bucket  : se-data-dump-2025-3
IAM role   : arn:aws:iam::650720751298:role/LabRole


---
## 1. DynamoDB table: `posts_batch_manifest`

Partition key: `batch_id` (string).  
Fields written by Lambda: `s3_path`, `size_bytes`, `uploaded_at`, `status`.  
Phase 2 updates `status` → `"done"` after processing.

In [2]:
try:
    dynamodb_c.create_table(
        TableName=DYNAMODB_TABLE,
        KeySchema=[{"AttributeName": "batch_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "batch_id", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST",
    )
    print(f"Created DynamoDB table: {DYNAMODB_TABLE} — waiting for ACTIVE...")
    dynamodb_c.get_waiter("table_exists").wait(TableName=DYNAMODB_TABLE)
    print("Table is ACTIVE.")
except dynamodb_c.exceptions.ResourceInUseException:
    print(f"Table already exists: {DYNAMODB_TABLE}")

Table already exists: posts_batch_manifest


---
## 2. SQS queue: `posts-batch-queue`

`VisibilityTimeout=900s` is larger than one Spark batch's expected run time,
preventing duplicate processing if Phase 2 is slow.

In [3]:
try:
    resp = sqs.create_queue(
        QueueName=SQS_QUEUE_NAME,
        Attributes={"VisibilityTimeout": str(SQS_VISIBILITY_TIMEOUT)},
    )
    SQS_URL = resp["QueueUrl"]
    print(f"Created SQS queue: {SQS_QUEUE_NAME}")
except sqs.exceptions.QueueNameExists:
    SQS_URL = sqs.get_queue_url(QueueName=SQS_QUEUE_NAME)["QueueUrl"]
    print(f"Queue already exists: {SQS_QUEUE_NAME}")

SQS_ARN = sqs.get_queue_attributes(
    QueueUrl=SQS_URL, AttributeNames=["QueueArn"]
)["Attributes"]["QueueArn"]

print(f"SQS URL: {SQS_URL}")
print(f"SQS ARN: {SQS_ARN}")

Created SQS queue: posts-batch-queue
SQS URL: https://sqs.us-east-1.amazonaws.com/650720751298/posts-batch-queue
SQS ARN: arn:aws:sqs:us-east-1:650720751298:posts-batch-queue


---
## 3. Lambda: `batch-manifest-writer`

Reads `lambda_function.py` from the same directory, zips it, and deploys.
Idempotent: updates code + config if the function already exists.

In [4]:
with open("lambda_function.py") as f:
    LAMBDA_SOURCE = f.read()
print(f"lambda_function.py loaded ({len(LAMBDA_SOURCE)} chars).")

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", LAMBDA_SOURCE)
zip_bytes = buf.getvalue()

env_vars = {
    "BUCKET":         BUCKET,
    "DYNAMODB_TABLE": DYNAMODB_TABLE,
    "SQS_URL":        SQS_URL,
}

try:
    lam.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.11",
        Role=LAB_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=LAMBDA_TIMEOUT,
        Environment={"Variables": env_vars},
    )
    print(f"Created Lambda: {LAMBDA_NAME}")
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=LAMBDA_NAME, ZipFile=zip_bytes)
    lam.get_waiter("function_updated").wait(FunctionName=LAMBDA_NAME)
    lam.update_function_configuration(
        FunctionName=LAMBDA_NAME,
        Environment={"Variables": env_vars},
        Timeout=LAMBDA_TIMEOUT,
    )
    print(f"Updated Lambda: {LAMBDA_NAME}")

lam.get_waiter("function_active").wait(FunctionName=LAMBDA_NAME)
print(f"Lambda Active. ARN: {LAMBDA_ARN}")

lambda_function.py loaded (1490 chars).
Updated Lambda: batch-manifest-writer
Lambda Active. ARN: arn:aws:lambda:us-east-1:650720751298:function:batch-manifest-writer


---
## 4. S3 bucket + Lambda trigger

Step 4a creates the S3 bucket (idempotent — safe to re-run if Phase 1 already created it).  
Step 4b grants S3 permission to invoke Lambda.  
Step 4c configures the bucket notification (prefix + suffix filter).

In [5]:
# 4a: Create S3 bucket (must exist before put_bucket_notification_configuration)
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Created S3 bucket: s3://{BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists (owned by you): s3://{BUCKET}")
except Exception as e:
    if "BucketAlreadyExists" in str(e):
        print(f"Bucket already exists: s3://{BUCKET}")
    else:
        raise

# 4b: Grant S3 permission to invoke Lambda
try:
    lam.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="allow-s3-invoke",
        Action="lambda:InvokeFunction",
        Principal="s3.amazonaws.com",
        SourceArn=f"arn:aws:s3:::{BUCKET}",
    )
    print("S3 invoke permission granted.")
except lam.exceptions.ResourceConflictException:
    print("Permission already exists — OK.")

# 4c: Configure S3 bucket notification
s3.put_bucket_notification_configuration(
    Bucket=BUCKET,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [{
            "LambdaFunctionArn": LAMBDA_ARN,
            "Events": ["s3:ObjectCreated:*"],
            "Filter": {"Key": {"FilterRules": [
                {"Name": "prefix", "Value": "raw/stackoverflow/posts_batches/"},
                {"Name": "suffix", "Value": ".xml.gz"},
            ]}},
        }],
    },
)
print(f"S3 trigger configured: s3://{BUCKET}/.../posts_batches/*.xml.gz → {LAMBDA_NAME}")

Created S3 bucket: s3://se-data-dump-2025-3
Permission already exists — OK.
S3 trigger configured: s3://se-data-dump-2025-3/.../posts_batches/*.xml.gz → batch-manifest-writer


---
## 5. SNS topic + email subscription

Phase 2 publishes to this topic after all batches are processed.
**Check email and confirm the subscription before the full run.**

In [6]:
topic = sns.create_topic(Name=SNS_TOPIC_NAME)
SNS_TOPIC_ARN = topic["TopicArn"]
print(f"SNS topic ARN: {SNS_TOPIC_ARN}")

resp = sns.subscribe(
    TopicArn=SNS_TOPIC_ARN,
    Protocol="email",
    Endpoint=NOTIFY_EMAIL,
)
print(f"Subscription pending confirmation at: {NOTIFY_EMAIL}")
print(f"SubscriptionArn: {resp['SubscriptionArn']}")
print("→ Check your inbox and click Confirm Subscription before the full run.")

SNS topic ARN: arn:aws:sns:us-east-1:650720751298:batch-etl-complete
Subscription pending confirmation at: graceeehgq@gmail.com
SubscriptionArn: pending confirmation
→ Check your inbox and click Confirm Subscription before the full run.


---
## 6. Verify all resources

In [7]:
print("=== Resource Verification ===")

ddb_status = dynamodb_c.describe_table(TableName=DYNAMODB_TABLE)["Table"]["TableStatus"]
print(f"DynamoDB {DYNAMODB_TABLE}: {ddb_status}")

sqs_url_check = sqs.get_queue_url(QueueName=SQS_QUEUE_NAME)["QueueUrl"]
print(f"SQS {SQS_QUEUE_NAME}: {sqs_url_check}")

lambda_state = lam.get_function(FunctionName=LAMBDA_NAME)["Configuration"]["State"]
print(f"Lambda {LAMBDA_NAME}: {lambda_state}")

cfg = s3.get_bucket_notification_configuration(Bucket=BUCKET)
n_triggers = len(cfg.get("LambdaFunctionConfigurations", []))
print(f"S3 Lambda triggers: {n_triggers}")

topics = sns.list_topics()["Topics"]
matched = [t for t in topics if SNS_TOPIC_NAME in t["TopicArn"]]
print(f"SNS {SNS_TOPIC_NAME}: {'found' if matched else 'NOT FOUND'}")

=== Resource Verification ===
DynamoDB posts_batch_manifest: ACTIVE
SQS posts-batch-queue: https://sqs.us-east-1.amazonaws.com/650720751298/posts-batch-queue
Lambda batch-manifest-writer: Active
S3 Lambda triggers: 1
SNS batch-etl-complete: found


---
## 7. Lambda unit test (no Phase 1 needed)

Invoke Lambda directly with a fake S3 event.  
Verifies: DynamoDB record created, SQS message enqueued.

In [8]:
TEST_BATCH_ID = "posts_batch_00000.xml.gz"
TEST_KEY      = f"raw/stackoverflow/posts_batches/{TEST_BATCH_ID}"

fake_event = {
    "Records": [{
        "s3": {"object": {"key": TEST_KEY, "size": 12345}},
        "eventTime": "2025-05-01T00:00:00.000Z",
    }]
}

resp = lam.invoke(
    FunctionName=LAMBDA_NAME,
    Payload=json.dumps(fake_event),
)
result = json.loads(resp["Payload"].read())
print(f"Lambda response: {result}")

# Verify DynamoDB record
ddb = boto3.resource("dynamodb", region_name=REGION)
item = ddb.Table(DYNAMODB_TABLE).get_item(
    Key={"batch_id": TEST_BATCH_ID}
).get("Item")
print(f"DynamoDB item: {item}")

# Verify SQS message exists
msgs = sqs.receive_message(
    QueueUrl=SQS_URL, MaxNumberOfMessages=1, WaitTimeSeconds=3
).get("Messages", [])
if msgs:
    body = json.loads(msgs[0]["Body"])
    print(f"SQS message: {body}")
    # Delete the test message so it doesn't interfere with real runs
    sqs.delete_message(QueueUrl=SQS_URL, ReceiptHandle=msgs[0]["ReceiptHandle"])
    print("Test SQS message deleted.")
else:
    print("[WARN] No SQS message found — check Lambda logs.")

Lambda response: {'statusCode': 200}
DynamoDB item: {'batch_id': 'posts_batch_00000.xml.gz', 's3_path': 's3://se-data-dump-2025-3/raw/stackoverflow/posts_batches/posts_batch_00000.xml.gz', 'size_bytes': Decimal('12345'), 'status': 'uploaded', 'uploaded_at': '2025-05-01T00:00:00.000Z'}
SQS message: {'batch_id': 'posts_batch_00007.xml.gz', 's3_path': 's3://se-data-dump-2025-3/raw/stackoverflow/posts_batches/posts_batch_00007.xml.gz'}
Test SQS message deleted.


---
## 8. Print Phase 2 run commands

Copy-paste these into the EMR master node after confirming the SNS subscription.

In [9]:
print("=== Phase 2 --mode sqs (copy to EMR) ===")
print(f"export SQS_URL='{SQS_URL}'")
print(f"export SNS_TOPIC_ARN='{SNS_TOPIC_ARN}'")
print()
print("spark-submit scripts/build_posts_cqi_batches.py \\")
print("  --mode          sqs \\")
print("  --sqs-url       $SQS_URL \\")
print("  --sns-topic-arn $SNS_TOPIC_ARN \\")
print(f"  --users-xml     s3://{BUCKET}/raw/stackoverflow/Users.xml \\")
print(f"  --postlinks-xml s3://{BUCKET}/raw/stackoverflow/PostLinks.xml \\")
print(f"  --output        s3://{BUCKET}/posts_cqi_with_tier/ \\")
print(f"  --checkpoint    s3://{BUCKET}/checkpoints/posts_cqi_batches/")

=== Phase 2 --mode sqs (copy to EMR) ===
export SQS_URL='https://sqs.us-east-1.amazonaws.com/650720751298/posts-batch-queue'
export SNS_TOPIC_ARN='arn:aws:sns:us-east-1:650720751298:batch-etl-complete'

spark-submit scripts/build_posts_cqi_batches.py \
  --mode          sqs \
  --sqs-url       $SQS_URL \
  --sns-topic-arn $SNS_TOPIC_ARN \
  --users-xml     s3://se-data-dump-2025-3/raw/stackoverflow/Users.xml \
  --postlinks-xml s3://se-data-dump-2025-3/raw/stackoverflow/PostLinks.xml \
  --output        s3://se-data-dump-2025-3/posts_cqi_with_tier/ \
  --checkpoint    s3://se-data-dump-2025-3/checkpoints/posts_cqi_batches/
